In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, roc_auc_score, roc_curve)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv('website\survey_cleaned.csv')
print(f"Loaded: {df.shape}")

# Target: Binary classification — Job vs Higher Studies 
# Keep only the two most common classes for a clean binary problem
binary = df[df['post_grad'].isin(['Job/Placement','Higher Studies'])].copy()
binary['target'] = (binary['post_grad'] == 'Higher Studies').astype(int)
print(f"Binary subset (Job vs Higher Studies): {binary.shape}")
print(f"Class distribution:\n{binary['target'].value_counts()}")
print(f"  0 = Job/Placement, 1 = Higher Studies")

# Feature Engineering 
# CGPA bucket → ordinal
cgpa_ord = {'Below 6':1,'6-7':2,'7-8':3,'8-9':4,'9-10':5}
binary['cgpa_num'] = binary['cgpa_bucket'].map(cgpa_ord).fillna(3)

# Gender binary
binary['is_male'] = (binary['gender'] == 'Male').astype(int)

# Core career interest → ordinal
core_ord = {'No definitely not':1,'No not really':2,'No probably not':2,
            'Neutral':3,'Yes somewhat':4,'Yes strongly':5,'Yes definitely':5}
if 'core_career' in binary.columns:
    binary['core_num'] = binary['core_career'].map(core_ord).fillna(3)
elif 'core_interest' in binary.columns:
    binary['core_num'] = binary['core_interest'].map(core_ord).fillna(3)
else:
    binary['core_num'] = 3

# Parent edu → ordinal
pared_ord = {
    'Neither parent graduated (School or below)': 1,
    'Neither parent graduated': 1,
    'One parent graduated': 2,
    'Both parents graduated (college+)': 3,
    'Prefer not to say': 2
}
binary['parent_edu_num'] = binary['parent_edu'].map(pared_ord).fillna(2)

# Features
FEATURES = [
    'cgpa_num',          # CGPA range (ordinal)
    'cgpa_sat',          # CGPA satisfaction (1-5)
    'class_imp',         # Class attendance importance
    'elective_sat',      # Elective satisfaction
    'prepared',          # Prepared for life rating
    'intern_help',       # Internship helpfulness
    'imp_interest',      # Job factor: interest
    'imp_salary',        # Job factor: salary
    'imp_location',      # Job factor: location
    'imp_culture',       # Job factor: work culture
    'brand_help',        # IIT brand perception
    'is_male',           # Gender
    'core_num',          # Core branch career interest
    'parent_edu_num',    # Parents' education
]

X = binary[FEATURES].fillna(binary[FEATURES].median())
y = binary['target']

print(f"\nFeature matrix: {X.shape}")
print(f"Features used: {FEATURES}")


Loaded: (97, 55)
Binary subset (Job vs Higher Studies): (71, 56)
Class distribution:
target
0    59
1    12
Name: count, dtype: int64
  0 = Job/Placement, 1 = Higher Studies

Feature matrix: (71, 14)
Features used: ['cgpa_num', 'cgpa_sat', 'class_imp', 'elective_sat', 'prepared', 'intern_help', 'imp_interest', 'imp_salary', 'imp_location', 'imp_culture', 'brand_help', 'is_male', 'core_num', 'parent_edu_num']


In [3]:

# Train / Test Split 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# Train Model
model = LogisticRegression(C=1.0, max_iter=1000, random_state=42, solver='lbfgs')
model.fit(X_train_s, y_train)

# Evaluation
y_pred = model.predict(X_test_s)
y_prob = model.predict_proba(X_test_s)[:, 1]


In [4]:

print("\n" + "="*55)
print("LOGISTIC REGRESSION RESULTS")
print("="*55)
print(f"\nTest Accuracy : {model.score(X_test_s, y_test):.3f}")
print(f"ROC-AUC Score : {roc_auc_score(y_test, y_prob):.3f}")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, scaler.transform(X), y, cv=cv, scoring='accuracy')
print(f"5-Fold CV Accuracy : {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Job/Placement','Higher Studies']))

# Feature Coefficients
coefs = pd.Series(model.coef_[0], index=FEATURES).sort_values(key=abs, ascending=False)
print("\nTop Feature Coefficients (absolute):")
print(coefs.to_string())



LOGISTIC REGRESSION RESULTS

Test Accuracy : 0.722
ROC-AUC Score : 0.622
5-Fold CV Accuracy : 0.774 ± 0.055

Classification Report:
                precision    recall  f1-score   support

 Job/Placement       0.81      0.87      0.84        15
Higher Studies       0.00      0.00      0.00         3

      accuracy                           0.72        18
     macro avg       0.41      0.43      0.42        18
  weighted avg       0.68      0.72      0.70        18


Top Feature Coefficients (absolute):
class_imp         1.167983
imp_interest     -0.764206
imp_culture       0.634844
cgpa_sat         -0.625365
cgpa_num          0.454087
is_male          -0.450440
imp_location     -0.365508
elective_sat     -0.248676
core_num         -0.133918
parent_edu_num    0.095996
brand_help       -0.091548
intern_help       0.067279
imp_salary       -0.053256
prepared          0.030819


In [5]:

# Visualisations 
PALETTE = ['#4361ee','#f72585','#4cc9f0','#06d6a0']
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.patch.set_facecolor('#0a0f1e')
for ax in axes:
    ax.set_facecolor('#111827')
    ax.tick_params(colors='#94a3b8')
    ax.xaxis.label.set_color('#94a3b8')
    ax.yaxis.label.set_color('#94a3b8')
    ax.title.set_color('#e2e8f0')
    for spine in ax.spines.values(): spine.set_edgecolor('#1e2d45')

# 1. Confusion matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['Job', 'Higher\nStudies'])
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix', pad=10)
axes[0].set_facecolor('#111827')

# 2. Feature importance (coefficients)
top8 = coefs.head(8)
colors = ['#f72585' if v > 0 else '#4361ee' for v in top8.values]
axes[1].barh(top8.index, top8.values, color=colors)
axes[1].axvline(0, color='#94a3b8', linewidth=0.8, linestyle='--')
axes[1].set_title('Top Feature Coefficients', pad=10)
axes[1].set_xlabel('Coefficient Value')
axes[1].invert_yaxis()

# 3. ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
auc_val = roc_auc_score(y_test, y_prob)
axes[2].plot(fpr, tpr, color='#4cc9f0', lw=2, label=f'AUC = {auc_val:.2f}')
axes[2].plot([0,1],[0,1], color='#94a3b8', linestyle='--', lw=1)
axes[2].fill_between(fpr, tpr, alpha=0.1, color='#4cc9f0')
axes[2].set_xlabel('False Positive Rate')
axes[2].set_ylabel('True Positive Rate')
axes[2].set_title('ROC Curve', pad=10)
axes[2].legend(loc='lower right', facecolor='#1a2235', labelcolor='#e2e8f0')

plt.suptitle('Logistic Regression: Predicting Job vs Higher Studies',
             color='#f8fafc', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('charts\model_results.png', dpi=150, bbox_inches='tight',
            facecolor='#0a0f1e')
plt.close()

print("\n✅ Model plots saved → charts\model_results.png")


FileNotFoundError: [Errno 2] No such file or directory: 'charts\\model_results.png'

In [6]:

# Model Interpretation Summary 
print("\n" + "="*55)
print("MODEL INTERPRETATION")
print("="*55)
print("""
Objective: Predict whether a graduating student will pursue
higher studies (1) or job/placement (0) after graduation.

Features: CGPA range, satisfaction ratings, career preferences,
demographic factors (gender, parents' education, brand perception).

Key Findings:
- Accuracy: {:.1f}% on held-out test set
- AUC-ROC:  {:.2f} (moderate discriminative power)
- Cross-val: {:.1f}% ± {:.1f}% (consistent across folds)

Influential Predictors (by coefficient magnitude):
{}

Note: With only {} samples in the binary subset, the model
has limited statistical power. Results should be interpreted
as indicative trends rather than definitive predictions.
The class imbalance (Job >> Higher Studies) also affects
performance on the minority class.
""".format(
    model.score(X_test_s, y_test)*100,
    auc_val,
    cv_scores.mean()*100,
    cv_scores.std()*100,
    coefs.head(5).to_string(),
    len(binary)
))


MODEL INTERPRETATION

Objective: Predict whether a graduating student will pursue
higher studies (1) or job/placement (0) after graduation.

Features: CGPA range, satisfaction ratings, career preferences,
demographic factors (gender, parents' education, brand perception).

Key Findings:
- Accuracy: 72.2% on held-out test set
- AUC-ROC:  0.62 (moderate discriminative power)
- Cross-val: 77.4% ± 5.5% (consistent across folds)

Influential Predictors (by coefficient magnitude):
class_imp       1.167983
imp_interest   -0.764206
imp_culture     0.634844
cgpa_sat       -0.625365
cgpa_num        0.454087

Note: With only 71 samples in the binary subset, the model
has limited statistical power. Results should be interpreted
as indicative trends rather than definitive predictions.
The class imbalance (Job >> Higher Studies) also affects
performance on the minority class.

